# Notebook 10 — Baseline Comparisons

**Project:** Calibrated and Stability-Aware Explainable Intrusion Detection
**Author:** Md Anas Biswas, University of Portsmouth
**Stage:** 10 of 10

## Scope

Compares this work's headline numbers to:
1. **Vanilla baselines** (no class weight, no SMOTE) — reproduced from scratch on NSL-KDD + CIC-IDS2017 here
2. **Published baselines** — numbers cited from recent literature (Arreche et al. 2024 E-XAI, Mahbooba 2021)

## Important limitations

Cross-paper reproduction is rarely exact: published papers often omit preprocessing details, exact hyperparameters, or use slightly different data splits. The vanilla baselines we train here are **representative of standard practice in the literature**, not exact reproductions of any single paper. This is documented clearly in the deliverable.

## What this notebook does

- Trains vanilla RF and XGBoost (no balancing tricks) on NSL-KDD and CIC-IDS2017
- Computes Acc / Macro-F1 / MCC / per-class F1
- Builds a side-by-side comparison table with published numbers
- Saves baseline metrics for paper Related Work

## Output

```
results/baselines/
├── vanilla_rf_nsl.pkl, vanilla_rf_cic.pkl
├── vanilla_xgb_nsl.pkl, vanilla_xgb_cic.pkl
├── baseline_metrics.json
results/tables/
└── baseline_comparison.csv
results/paper_tables/
└── paper_table_8_baselines.csv
```

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
Already up to date.
✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import pandas as pd
import json, pickle, time
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
import xgboost as xgb

SEED = 42
np.random.seed(SEED)

# Paths
NSL = Path(REPO) / 'data' / 'processed' / 'nsl_kdd'
CIC = Path(REPO) / 'data' / 'processed' / 'cic_ids2017'
BASE_DIR = Path(REPO) / 'results' / 'baselines'
BASE_DIR.mkdir(parents=True, exist_ok=True)
TABLES = Path(REPO) / 'results' / 'tables'
PAPER_TABLES = Path(REPO) / 'results' / 'paper_tables'

# Load both datasets
data = {}
for name, path in [('NSL-KDD', NSL), ('CIC-IDS2017', CIC)]:
    data[name] = {
        'X_train': np.load(path / 'X_train.npy'),
        'X_test':  np.load(path / 'X_test.npy'),
        'y_train_b': np.load(path / 'y_train_binary.npy'),
        'y_test_b':  np.load(path / 'y_test_binary.npy'),
        'y_train_5': np.load(path / 'y_train_5class.npy'),
        'y_test_5':  np.load(path / 'y_test_5class.npy'),
    }
    print(f'{name}: train={data[name]["X_train"].shape}, test={data[name]["X_test"].shape}')

NSL-KDD: train=(125973, 122), test=(22544, 122)
CIC-IDS2017: train=(160024, 78), test=(40007, 78)


---
## Step 1 — Evaluation helper

In [3]:
def evaluate(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_weighted': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'mcc': float(matthews_corrcoef(y_true, y_pred)),
    }

ALL_METRICS = {}

---
## Step 2 — Vanilla Random Forest (no class weight, no SMOTE)

Hyperparameters: n_estimators=100, default sklearn settings — representative of standard practice (e.g. Tavallaee 2009, many follow-up papers).

In [4]:
for ds in ['NSL-KDD', 'CIC-IDS2017']:
    d = data[ds]

    # Binary
    print(f'\n--- Vanilla RF on {ds} (binary) ---')
    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
    rf.fit(d['X_train'], d['y_train_b'])
    y_pred = rf.predict(d['X_test'])
    m = evaluate(d['y_test_b'], y_pred)
    print(f'  Trained in {time.time()-t0:.1f}s')
    print(f'  Acc={m["accuracy"]:.4f}  MacroF1={m["f1_macro"]:.4f}  MCC={m["mcc"]:.4f}')
    ALL_METRICS[f'vanilla_rf_{ds.replace("-","").lower()}_binary'] = m
    with open(BASE_DIR / f'vanilla_rf_{ds.replace("-","").lower()}_binary.pkl', 'wb') as f:
        pickle.dump(rf, f)

    # 5-class
    print(f'\n--- Vanilla RF on {ds} (5-class) ---')
    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
    rf.fit(d['X_train'], d['y_train_5'])
    y_pred = rf.predict(d['X_test'])
    m = evaluate(d['y_test_5'], y_pred)
    print(f'  Trained in {time.time()-t0:.1f}s')
    print(f'  Acc={m["accuracy"]:.4f}  MacroF1={m["f1_macro"]:.4f}  MCC={m["mcc"]:.4f}')
    ALL_METRICS[f'vanilla_rf_{ds.replace("-","").lower()}_5class'] = m
    with open(BASE_DIR / f'vanilla_rf_{ds.replace("-","").lower()}_5class.pkl', 'wb') as f:
        pickle.dump(rf, f)


--- Vanilla RF on NSL-KDD (binary) ---
  Trained in 15.2s
  Acc=0.7648  MacroF1=0.7635  MCC=0.5993

--- Vanilla RF on NSL-KDD (5-class) ---
  Trained in 16.2s
  Acc=0.7502  MacroF1=0.4819  MCC=0.6391

--- Vanilla RF on CIC-IDS2017 (binary) ---
  Trained in 61.0s
  Acc=0.9981  MacroF1=0.9969  MCC=0.9939

--- Vanilla RF on CIC-IDS2017 (5-class) ---
  Trained in 61.1s
  Acc=0.9981  MacroF1=0.9588  MCC=0.9944


---
## Step 3 — Vanilla XGBoost

Hyperparameters: n_estimators=100, default depth/lr — standard 'no-tuning' baseline.

In [ ]:
for ds in ['NSL-KDD', 'CIC-IDS2017']:
    d = data[ds]

    # Binary
    print(f'\n--- Vanilla XGB on {ds} (binary) ---')
    t0 = time.time()
    xb = xgb.XGBClassifier(n_estimators=100, tree_method='hist', random_state=SEED, n_jobs=-1,
                            objective='binary:logistic', eval_metric='logloss')
    xb.fit(d['X_train'], d['y_train_b'])
    y_pred = xb.predict(d['X_test'])
    m = evaluate(d['y_test_b'], y_pred)
    print(f'  Trained in {time.time()-t0:.1f}s')
    print(f'  Acc={m["accuracy"]:.4f}  MacroF1={m["f1_macro"]:.4f}  MCC={m["mcc"]:.4f}')
    ALL_METRICS[f'vanilla_xgb_{ds.replace("-","").lower()}_binary'] = m
    with open(BASE_DIR / f'vanilla_xgb_{ds.replace("-","").lower()}_binary.pkl', 'wb') as f:
        pickle.dump(xb, f)

    # 5-class
    print(f'\n--- Vanilla XGB on {ds} (5-class) ---')
    t0 = time.time()
    xb = xgb.XGBClassifier(n_estimators=100, tree_method='hist', random_state=SEED, n_jobs=-1,
                            objective='multi:softprob', num_class=5, eval_metric='mlogloss')
    xb.fit(d['X_train'], d['y_train_5'])
    y_pred = xb.predict(d['X_test'])
    m = evaluate(d['y_test_5'], y_pred)
    print(f'  Trained in {time.time()-t0:.1f}s')
    print(f'  Acc={m["accuracy"]:.4f}  MacroF1={m["f1_macro"]:.4f}  MCC={m["mcc"]:.4f}')
    ALL_METRICS[f'vanilla_xgb_{ds.replace("-","").lower()}_5class'] = m
    with open(BASE_DIR / f'vanilla_xgb_{ds.replace("-","").lower()}_5class.pkl', 'wb') as f:
        pickle.dump(xb, f)

with open(BASE_DIR / 'baseline_metrics.json', 'w') as f:
    json.dump(ALL_METRICS, f, indent=2)
print(f'\n✓ Saved {len(ALL_METRICS)} baseline models')


--- Vanilla XGB on NSL-KDD (binary) ---
  Trained in 4.5s
  Acc=0.7879  MacroF1=0.7874  MCC=0.6326

--- Vanilla XGB on NSL-KDD (5-class) ---


---
## Step 4 — Comparison table (vanilla vs ours vs literature)

In [ ]:
# Load our framework's metrics
with open(Path(REPO) / 'models' / 'nsl_kdd' / 'metrics.json') as f:
    nsl_metrics = json.load(f)
with open(Path(REPO) / 'models' / 'cic_ids2017' / 'metrics.json') as f:
    cic_metrics = json.load(f)

# Build comparison rows
rows = []

# --- NSL-KDD ---
# Published literature reference values (cited, not reproduced)
PUBLISHED_REFS = [
    {'Source': 'Tavallaee et al. 2009 (NSL-KDD intro)', 'Dataset': 'NSL-KDD', 'Model': 'NB-Tree',
     'Accuracy': 0.811, 'Macro_F1': None, 'MCC': None, 'Notes': 'Original NSL-KDD paper'},
    {'Source': 'Mahbooba 2021', 'Dataset': 'NSL-KDD', 'Model': 'Decision Tree',
     'Accuracy': 0.789, 'Macro_F1': None, 'MCC': None, 'Notes': 'XAI-IDS with DT'},
    {'Source': 'Arreche 2024 (E-XAI)', 'Dataset': 'NSL-KDD', 'Model': 'RF (binary)',
     'Accuracy': 0.768, 'Macro_F1': None, 'MCC': None, 'Notes': 'E-XAI baseline'},
]
for ref in PUBLISHED_REFS:
    rows.append(ref)

# Vanilla baselines (our reproductions)
rows.append({'Source': 'Vanilla (ours)',  'Dataset': 'NSL-KDD',     'Model': 'RF (binary)',
             'Accuracy': ALL_METRICS['vanilla_rf_nslkdd_binary']['accuracy'],
             'Macro_F1': ALL_METRICS['vanilla_rf_nslkdd_binary']['f1_macro'],
             'MCC': ALL_METRICS['vanilla_rf_nslkdd_binary']['mcc'],
             'Notes': 'No class weight, no SMOTE'})
rows.append({'Source': 'Vanilla (ours)',  'Dataset': 'NSL-KDD',     'Model': 'RF (5-class)',
             'Accuracy': ALL_METRICS['vanilla_rf_nslkdd_5class']['accuracy'],
             'Macro_F1': ALL_METRICS['vanilla_rf_nslkdd_5class']['f1_macro'],
             'MCC': ALL_METRICS['vanilla_rf_nslkdd_5class']['mcc'],
             'Notes': 'No class weight, no SMOTE'})
rows.append({'Source': 'Vanilla (ours)',  'Dataset': 'NSL-KDD',     'Model': 'XGB (binary)',
             'Accuracy': ALL_METRICS['vanilla_xgb_nslkdd_binary']['accuracy'],
             'Macro_F1': ALL_METRICS['vanilla_xgb_nslkdd_binary']['f1_macro'],
             'MCC': ALL_METRICS['vanilla_xgb_nslkdd_binary']['mcc'],
             'Notes': 'No class weight'})
rows.append({'Source': 'Vanilla (ours)',  'Dataset': 'NSL-KDD',     'Model': 'XGB (5-class)',
             'Accuracy': ALL_METRICS['vanilla_xgb_nslkdd_5class']['accuracy'],
             'Macro_F1': ALL_METRICS['vanilla_xgb_nslkdd_5class']['f1_macro'],
             'MCC': ALL_METRICS['vanilla_xgb_nslkdd_5class']['mcc'],
             'Notes': 'No class weight'})

# Our framework's numbers
for name in ['rf_binary_cw', 'xgb_binary_cw', 'rf_5class_smote', 'xgb_5class_smote']:
    m = nsl_metrics[name]
    pretty_name = name.replace('_', ' ').replace('cw', '(class-weighted)').replace('smote', '(SMOTE)')
    rows.append({
        'Source': 'Our framework',
        'Dataset': 'NSL-KDD',
        'Model': pretty_name,
        'Accuracy': m['accuracy'],
        'Macro_F1': m['f1_macro'],
        'MCC': m['mcc'],
        'Notes': 'With calibration + SHAP + SCTS-v2',
    })

# --- CIC-IDS2017 ---
rows.append({'Source': 'Vanilla (ours)', 'Dataset': 'CIC-IDS2017', 'Model': 'RF (binary)',
             'Accuracy': ALL_METRICS['vanilla_rf_cicids2017_binary']['accuracy'],
             'Macro_F1': ALL_METRICS['vanilla_rf_cicids2017_binary']['f1_macro'],
             'MCC': ALL_METRICS['vanilla_rf_cicids2017_binary']['mcc'],
             'Notes': 'No class weight'})
rows.append({'Source': 'Vanilla (ours)', 'Dataset': 'CIC-IDS2017', 'Model': 'RF (5-class)',
             'Accuracy': ALL_METRICS['vanilla_rf_cicids2017_5class']['accuracy'],
             'Macro_F1': ALL_METRICS['vanilla_rf_cicids2017_5class']['f1_macro'],
             'MCC': ALL_METRICS['vanilla_rf_cicids2017_5class']['mcc'],
             'Notes': 'No class weight'})
rows.append({'Source': 'Vanilla (ours)', 'Dataset': 'CIC-IDS2017', 'Model': 'XGB (binary)',
             'Accuracy': ALL_METRICS['vanilla_xgb_cicids2017_binary']['accuracy'],
             'Macro_F1': ALL_METRICS['vanilla_xgb_cicids2017_binary']['f1_macro'],
             'MCC': ALL_METRICS['vanilla_xgb_cicids2017_binary']['mcc'],
             'Notes': 'No class weight'})
rows.append({'Source': 'Vanilla (ours)', 'Dataset': 'CIC-IDS2017', 'Model': 'XGB (5-class)',
             'Accuracy': ALL_METRICS['vanilla_xgb_cicids2017_5class']['accuracy'],
             'Macro_F1': ALL_METRICS['vanilla_xgb_cicids2017_5class']['f1_macro'],
             'MCC': ALL_METRICS['vanilla_xgb_cicids2017_5class']['mcc'],
             'Notes': 'No class weight'})

for name in ['rf_binary_cw', 'xgb_binary_cw', 'rf_5class_smote', 'xgb_5class_smote']:
    m = cic_metrics[name]
    pretty_name = name.replace('_', ' ').replace('cw', '(class-weighted)').replace('smote', '(SMOTE)')
    rows.append({
        'Source': 'Our framework',
        'Dataset': 'CIC-IDS2017',
        'Model': pretty_name,
        'Accuracy': m['accuracy'],
        'Macro_F1': m['f1_macro'],
        'MCC': m['mcc'],
        'Notes': 'With calibration + SHAP + SCTS-v2',
    })

df = pd.DataFrame(rows)
df.to_csv(TABLES / 'baseline_comparison.csv', index=False)
df.to_csv(PAPER_TABLES / 'paper_table_8_baselines.csv', index=False)

print('BASELINE COMPARISON')
print('=' * 110)
print(df.to_string(index=False, float_format='%.4f'))
print('=' * 110)

---
## Step 5 — Synthesis: how do our numbers compare?

Quick analysis paragraph for the paper.

In [ ]:
# Compute a few comparison statistics
ours_nsl_rf_b = nsl_metrics['rf_binary_cw']['accuracy']
vanilla_nsl_rf_b = ALL_METRICS['vanilla_rf_nslkdd_binary']['accuracy']
lit_nsl_rf_b = 0.768  # Arreche

ours_nsl_xgb_5 = nsl_metrics['xgb_5class_smote']['f1_macro']
vanilla_nsl_xgb_5 = ALL_METRICS['vanilla_xgb_nslkdd_5class']['f1_macro']

print('PAPER-READY SYNTHESIS:')
print('-' * 70)
print(f'\nNSL-KDD RF (binary):')
print(f'  Arreche 2024 (cited):   Acc = {lit_nsl_rf_b:.3f}')
print(f'  Vanilla RF (ours):       Acc = {vanilla_nsl_rf_b:.4f}')
print(f'  Our framework RF:        Acc = {ours_nsl_rf_b:.4f}')
print(f'\n  → Our framework: {"in line with" if abs(ours_nsl_rf_b - lit_nsl_rf_b) < 0.03 else "departs from"} published baseline')

print(f'\nNSL-KDD XGB (5-class Macro-F1):')
print(f'  Vanilla XGB:        F1m = {vanilla_nsl_xgb_5:.4f}')
print(f'  Our framework XGB:  F1m = {ours_nsl_xgb_5:.4f}')
improvement = (ours_nsl_xgb_5 - vanilla_nsl_xgb_5) / vanilla_nsl_xgb_5 * 100
print(f'  → SMOTE improvement: {improvement:+.1f}%')

print('\n' + '-' * 70)

In [ ]:
# Commit
import os
os.chdir(REPO)
!git add notebooks/10_baselines.ipynb
!git add results/baselines/baseline_metrics.json
!git add results/tables/baseline_comparison.csv
!git add results/paper_tables/paper_table_8_baselines.csv
!git status --short
!git commit -m 'Notebook 10: vanilla baselines + literature comparison (trees only, CPU)'
!git push

## Summary

**Done:**
- ✓ Vanilla RF + XGBoost trained on both NSL-KDD and CIC-IDS2017 (no class weight, no SMOTE)
- ✓ Side-by-side comparison with our framework's class-weighted/SMOTE results
- ✓ Published literature numbers included for context
- ✓ Single comparison CSV for paper's Related Work section

**Known limitations:**
- Vanilla baselines are *representative* of standard practice, not exact reproductions of any single paper
- Published reference numbers (Arreche, Mahbooba, Tavallaee) are cited, not reproduced — they may use slightly different preprocessing
- DNN baselines not included (would require GPU; ran in parallel with main GPU notebook)

**Next:** deliverable folder assembly + README.
